In [0]:
from pathlib import Path
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

silver_schema = "cdr_silver.processed_mimic_iv"

current_path = Path.cwd().resolve()
repo_root = None

for candidate in [current_path] + list(current_path.parents):
    if (
        (candidate / "notebooks").exists()
        and (candidate / "config").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError("Could not locate repository root.")

output_path = (
    repo_root
    / "build"
    / "old_silver_cleaning_baseline.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def count_if(condition):
    return F.sum(
        F.when(condition, 1).otherwise(0)
    )


def collect_metrics(df, expressions):
    row = df.agg(
        *[
            expression.alias(name)
            for name, expression in expressions.items()
        ]
    ).first()

    return {
        name: row[name]
        for name in expressions
    }


# ------------------------------------------------------------
# Load Silver tables
# ------------------------------------------------------------

table_names = [
    "admissions",
    "patients",
    "transfers",
    "services",
    "icustays",
    "icd_diagnoses",
    "d_icd_diagnoses",
    "icd_procedures",
    "chartevents",
    "d_items",
    "labevents",
    "d_labitems",
    "inputevents",
    "outputevents",
    "procedureevents"
]

silver = {}

snapshot = {
    "label": "OLD SILVER",
    "silver_schema": silver_schema,
    "schemas": {},
    "metrics": {}
}


for table in table_names:
    full_name = f"{silver_schema}.{table}"

    if not spark.catalog.tableExists(full_name):
        print(f"MISSING: {full_name}")
        continue

    df = spark.table(full_name)
    silver[table] = df

    snapshot["schemas"][table] = [
        {
            "name": field.name,
            "type": field.dataType.simpleString(),
            "nullable": field.nullable
        }
        for field in df.schema.fields
    ]

print(
    f"Loaded {len(silver)} of "
    f"{len(table_names)} Silver tables."
)


# ------------------------------------------------------------
# Gold 1 checks
# Admissions / transfers / ICU LOS
# ------------------------------------------------------------

admissions = silver["admissions"]

snapshot["metrics"]["admissions"] = collect_metrics(
    admissions,
    {
        "rows": F.count("*"),
        "missing_admittime": count_if(
            F.col("admittime").isNull()
        ),
        "missing_dischtime": count_if(
            F.col("dischtime").isNull()
        ),
        "negative_hospital_los": count_if(
            F.col("dischtime") < F.col("admittime")
        )
    }
)


icustays = silver["icustays"]

snapshot["metrics"]["icustays"] = collect_metrics(
    icustays,
    {
        "rows": F.count("*"),
        "missing_outtime": count_if(
            F.col("outtime").isNull()
        ),
        "missing_los": count_if(
            F.col("los").isNull()
        ),
        "negative_los": count_if(
            F.col("los") < 0
        ),
        "invalid_time_order": count_if(
            F.col("outtime").isNotNull()
            & (F.col("outtime") < F.col("intime"))
        )
    }
)


# ------------------------------------------------------------
# Gold 3 checks
# Transfer interval cleaning
# ------------------------------------------------------------

transfers = silver["transfers"]

snapshot["metrics"]["transfers_raw"] = collect_metrics(
    transfers,
    {
        "rows": F.count("*"),
        "missing_hadm_id": count_if(
            F.col("hadm_id").isNull()
        ),
        "missing_intime": count_if(
            F.col("intime").isNull()
        ),
        "missing_outtime": count_if(
            F.col("outtime").isNull()
        ),
        "null_careunit": count_if(
            F.col("careunit").isNull()
        ),
        "unknown_careunit": count_if(
            F.col("careunit") == "UNKNOWN"
        ),
        "invalid_time_order": count_if(
            F.col("intime").isNotNull()
            & F.col("outtime").isNotNull()
            & (F.col("outtime") < F.col("intime"))
        )
    }
)


movement_window = (
    Window
    .partitionBy("hadm_id")
    .orderBy(
        F.col("intime"),
        F.col("transfer_id")
    )
)

transfer_sequence = (
    transfers
    .withColumn(
        "_next_event_intime",
        F.lead("intime").over(movement_window)
    )
)

intervals = (
    transfer_sequence
    .filter(
        F.col("eventtype").isin("admit", "transfer")
        & F.col("hadm_id").isNotNull()
        & F.col("intime").isNotNull()
        & F.col("careunit").isNotNull()
        & (F.col("careunit") != "UNKNOWN")
    )
    .withColumn(
        "_outtime_inferred",
        F.col("outtime").isNull()
    )
    .withColumn(
        "_effective_outtime",
        F.coalesce(
            F.col("outtime"),
            F.col("_next_event_intime")
        )
    )
    .withColumn(
        "_interval_hours",
        (
            F.col("_effective_outtime").cast("long")
            - F.col("intime").cast("long")
        ) / 3600.0
    )
)

intervals_with_admission = (
    intervals
    .join(
        admissions.select(
            "hadm_id",
            "admittime",
            "dischtime"
        ),
        "hadm_id",
        "left"
    )
    .withColumn(
        "_end_after_discharge_hours",
        (
            F.col("_effective_outtime").cast("long")
            - F.col("dischtime").cast("long")
        ) / 3600.0
    )
)


snapshot["metrics"]["transfer_interval_rules"] = collect_metrics(
    intervals_with_admission,
    {
        "candidate_intervals": F.count("*"),

        "inferable_missing_outtime": count_if(
            F.col("outtime").isNull()
            & F.col("_next_event_intime").isNotNull()
        ),

        "missing_outtime_not_inferable": count_if(
            F.col("outtime").isNull()
            & F.col("_next_event_intime").isNull()
        ),

        "zero_duration_intervals": count_if(
            F.col("_interval_hours") == 0
        ),

        "negative_duration_intervals": count_if(
            F.col("_interval_hours") < 0
        ),

        "over_100_day_intervals": count_if(
            F.col("_interval_hours") > 24 * 100
        ),

        "extreme_timing_anomalies": count_if(
            F.col("dischtime").isNotNull()
            & (F.col("_interval_hours") > 24 * 100)
            & (
                F.col("_end_after_discharge_hours")
                > 24 * 30
            )
        )
    }
)


# ------------------------------------------------------------
# Gold 2 checks
# Vital-sign cleaning
# ------------------------------------------------------------

HEART_RATE_ID = 220045
RESP_RATE_ID = 220210
SPO2_ID = 220277
TEMP_C_ID = 223762
TEMP_F_ID = 223761

vital_ids = [
    HEART_RATE_ID,
    RESP_RATE_ID,
    SPO2_ID,
    TEMP_C_ID,
    TEMP_F_ID
]

vitals = (
    silver["chartevents"]
    .filter(F.col("itemid").isin(vital_ids))
    .withColumn(
        "_standard_value",
        F.when(
            F.col("itemid") == TEMP_F_ID,
            (F.col("valuenum") - 32)
            * (5.0 / 9.0)
        ).otherwise(F.col("valuenum"))
    )
)

snapshot["metrics"]["vital_cleaning"] = collect_metrics(
    vitals,
    {
        "selected_vital_rows": F.count("*"),

        "null_valuenum": count_if(
            F.col("valuenum").isNull()
        ),

        "invalid_heart_rate": count_if(
            (F.col("itemid") == HEART_RATE_ID)
            & F.col("_standard_value").isNotNull()
            & ~F.col("_standard_value").between(0, 400)
        ),

        "invalid_resp_rate": count_if(
            (F.col("itemid") == RESP_RATE_ID)
            & F.col("_standard_value").isNotNull()
            & ~F.col("_standard_value").between(0, 150)
        ),

        "invalid_spo2": count_if(
            (F.col("itemid") == SPO2_ID)
            & F.col("_standard_value").isNotNull()
            & ~F.col("_standard_value").between(0, 100)
        ),

        "invalid_temperature": count_if(
            F.col("itemid").isin(
                TEMP_C_ID,
                TEMP_F_ID
            )
            & F.col("_standard_value").isNotNull()
            & ~F.col("_standard_value").between(10, 50)
        ),

        "fahrenheit_temperature_rows": count_if(
            F.col("itemid") == TEMP_F_ID
        )
    }
)


# ------------------------------------------------------------
# Selected laboratory cleaning
# ------------------------------------------------------------

CREATININE_IDS = [50912, 52546]
HEMOGLOBIN_IDS = [51222]
WBC_IDS = [51300, 51301]
SODIUM_IDS = [50983, 52623]
POTASSIUM_IDS = [50971, 52610]
LACTATE_IDS = [50813, 52442, 53154]

selected_lab_ids = (
    CREATININE_IDS
    + HEMOGLOBIN_IDS
    + WBC_IDS
    + SODIUM_IDS
    + POTASSIUM_IDS
    + LACTATE_IDS
)

labs = (
    silver["labevents"]
    .filter(
        F.col("itemid").isin(selected_lab_ids)
    )
)

snapshot["metrics"]["lab_cleaning"] = collect_metrics(
    labs,
    {
        "selected_lab_rows": F.count("*"),

        "null_valuenum": count_if(
            F.col("valuenum").isNull()
        ),

        "abnormal_flag_rows": count_if(
            F.col("flag") == "abnormal"
        ),

        "unrecorded_flag_rows": count_if(
            F.col("flag").isNull()
        ),

        "invalid_lactate": count_if(
            F.col("itemid").isin(LACTATE_IDS)
            & F.col("valuenum").isNotNull()
            & ~F.col("valuenum").between(0, 50)
        )
    }
)


# ------------------------------------------------------------
# Fluid input cleaning
# ------------------------------------------------------------

inputs = silver["inputevents"]

unit = F.upper(
    F.trim(F.col("amountuom"))
)

amount_ml = (
    F.when(
        unit == "L",
        F.col("amount") * 1000
    )
    .when(
        unit.isin("ΜL", "UL"),
        F.col("amount") / 1000
    )
    .when(
        unit.isin("ML", "CM3"),
        F.col("amount")
    )
)

input_check = (
    inputs
    .withColumn(
        "_amount_ml",
        amount_ml
    )
    .withColumn(
        "_duration_minutes",
        (
            F.col("endtime").cast("long")
            - F.col("starttime").cast("long")
        ) / 60.0
    )
)

non_icu_input_ids = [
    226364,
    226367,
    226368,
    226377
]

snapshot["metrics"]["input_cleaning"] = collect_metrics(
    input_check,
    {
        "rows": F.count("*"),

        "missing_amount": count_if(
            F.col("amount").isNull()
        ),

        "unsupported_volume_unit": count_if(
            F.col("amount").isNotNull()
            & F.col("_amount_ml").isNull()
        ),

        "negative_standardised_volume": count_if(
            F.col("_amount_ml") < 0
        ),

        "invalid_duration": count_if(
            F.col("_duration_minutes") <= 0
        ),

        "above_50000_ml": count_if(
            F.col("_amount_ml") > 50000
        ),

        "non_icu_input_records": count_if(
            (
                F.col("ordercategoryname")
                == "16-Pre Admission/Non-ICU"
            )
            | F.col("itemid").isin(
                non_icu_input_ids
            )
        )
    }
)


# ------------------------------------------------------------
# Fluid output cleaning
# ------------------------------------------------------------

outputs = silver["outputevents"]

excluded_output_ids = [
    226632,
    226626,
    226627,
    226628,
    226629,
    226630,
    226631,
    226633,
    227487,
    227488,
    227489,
    226566
]

snapshot["metrics"]["output_cleaning"] = collect_metrics(
    outputs,
    {
        "rows": F.count("*"),

        "missing_value": count_if(
            F.col("value").isNull()
        ),

        "negative_output": count_if(
            F.col("value") < 0
        ),

        "above_50000_ml": count_if(
            F.col("value") > 50000
        ),

        "excluded_non_icu_output_rows": count_if(
            F.col("itemid").isin(
                excluded_output_ids
            )
        )
    }
)


# ------------------------------------------------------------
# Save baseline
# ------------------------------------------------------------

with output_path.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        snapshot,
        file,
        indent=2,
        default=str
    )


print()
print("OLD SILVER BASELINE CAPTURED")
print(f"Saved to: {output_path}")
print()

for section, metrics in snapshot["metrics"].items():
    print(f"\n{section.upper()}")
    for name, value in metrics.items():
        print(f"  {name}: {value}")